In [2]:
# ============================================================
# 02_visualizations.ipynb
# VISUALIZATION OF THE AIRPORT COMPLEX NETWORK
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from collections import Counter

import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.style.use("ggplot")

# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "../data"

FIGURES_DIR = BASE_DIR / "../figures"

FIGURES_DIR.mkdir(exist_ok=True)

print("Figures will be saved in:")
print(FIGURES_DIR)

# ============================================================
# LOAD NETWORK
# ============================================================

print("\n" + "=" * 60)
print("LOADING NETWORK")
print("=" * 60)

with open(DATA_DIR / "airport_network.gpickle", "rb") as f:
    G_main = pickle.load(f)

print(G_main)

print("\nNodes:", G_main.number_of_nodes())
print("Edges:", G_main.number_of_edges())

# ============================================================
# LOAD NODE ATTRIBUTES
# ============================================================

degrees = dict(G_main.degree())

communities = nx.get_node_attributes(
    G_main,
    "community"
)

betweenness = nx.get_node_attributes(
    G_main,
    "betweenness"
)

clustering = nx.get_node_attributes(
    G_main,
    "clustering"
)

# ============================================================
# 1. COMMUNITY NETWORK VISUALIZATION
# ============================================================

print("\n" + "=" * 60)
print("COMMUNITY VISUALIZATION")
print("=" * 60)

sample_nodes = list(G_main.nodes())[:300]

subgraph = G_main.subgraph(sample_nodes)

node_colors = [
    communities[node]
    for node in subgraph.nodes()
]

node_sizes = [
    degrees[node] * 10
    for node in subgraph.nodes()
]

plt.figure(figsize=(14,14))

pos = nx.spring_layout(
    subgraph,
    k=0.15,
    seed=42
)

nx.draw_networkx_nodes(
    subgraph,
    pos,
    node_color=node_colors,
    node_size=node_sizes,
    cmap=plt.cm.tab20,
    alpha=0.9
)

nx.draw_networkx_edges(
    subgraph,
    pos,
    alpha=0.15
)

plt.title(
    "Airport Network Communities",
    fontsize=20
)

plt.axis("off")

plt.savefig(
    FIGURES_DIR / "communities.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 2. DEGREE DISTRIBUTION
# ============================================================

print("\n" + "=" * 60)
print("DEGREE DISTRIBUTION")
print("=" * 60)

degree_values = list(degrees.values())

plt.figure(figsize=(8,5))

plt.hist(
    degree_values,
    bins=50,
    density=True
)

plt.xlabel("Degree k")
plt.ylabel("P(k)")

plt.title(
    "Degree Distribution",
    fontsize=18
)

plt.savefig(
    FIGURES_DIR / "degree_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 3. LOG-LOG DEGREE DISTRIBUTION
# ============================================================

degree_counts = Counter(degree_values)

k = np.array(list(degree_counts.keys()))

pk = np.array(
    list(degree_counts.values())
) / len(degree_values)

mask = (k > 0) & (pk > 0)

k_fit = k[mask]
pk_fit = pk[mask]

# ------------------------------------------------------------
# Power-law fit
# ------------------------------------------------------------

log_k = np.log10(k_fit)

log_pk = np.log10(pk_fit)

coeffs = np.polyfit(
    log_k,
    log_pk,
    1
)

gamma = -coeffs[0]

fit_line = (
    10 ** coeffs[1]
) * k_fit ** coeffs[0]

print("Estimated gamma:", gamma)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

plt.figure(figsize=(7,5))

plt.scatter(
    k_fit,
    pk_fit,
    label="Data"
)

plt.plot(
    k_fit,
    fit_line,
    linestyle="--",
    label=f"Fit γ={gamma:.2f}"
)

plt.xscale("log")
plt.yscale("log")

plt.xlabel("Degree k")
plt.ylabel("P(k)")

plt.title(
    "Degree Distribution (log-log)",
    fontsize=18
)

plt.legend()

plt.savefig(
    FIGURES_DIR / "loglog_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 4. CLUSTERING VS DEGREE
# ============================================================

print("\n" + "=" * 60)
print("CLUSTERING ANALYSIS")
print("=" * 60)

degrees_nodes = []
clustering_values = []

for node in G_main.nodes():

    degrees_nodes.append(
        degrees[node]
    )

    clustering_values.append(
        clustering[node]
    )

plt.figure(figsize=(7,5))

plt.scatter(
    degrees_nodes,
    clustering_values,
    alpha=0.5
)

plt.xscale("log")

plt.xlabel("Degree")
plt.ylabel("Clustering coefficient")

plt.title(
    "Clustering vs Degree",
    fontsize=18
)

plt.savefig(
    FIGURES_DIR / "clustering_vs_degree.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 5. SPECTRAL DENSITY
# ============================================================

print("\n" + "=" * 60)
print("SPECTRAL ANALYSIS")
print("=" * 60)

print("Computing adjacency matrix...")

A = nx.adjacency_matrix(G_main).todense()

print("Computing eigenvalues...")

eigenvalues = np.linalg.eigvals(A)

plt.figure(figsize=(7,5))

plt.hist(
    np.real(eigenvalues),
    bins=50
)

plt.xlabel("Eigenvalue")
plt.ylabel("Frequency")

plt.title(
    "Spectral Density",
    fontsize=18
)

plt.savefig(
    FIGURES_DIR / "spectral_density.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 6. ROBUSTNESS ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("ROBUSTNESS ANALYSIS")
print("=" * 60)

G_attack = G_main.copy()

top_nodes = sorted(
    G_attack.degree(),
    key=lambda x: x[1],
    reverse=True
)

sizes = []

for node, _ in top_nodes[:50]:

    G_attack.remove_node(node)

    if len(G_attack) == 0:
        break

    largest = max(
        nx.connected_components(G_attack),
        key=len
    )

    sizes.append(len(largest))

plt.figure(figsize=(7,5))

plt.plot(sizes)

plt.xlabel("Removed hubs")
plt.ylabel(
    "Largest connected component size"
)

plt.title(
    "Robustness Against Targeted Attacks",
    fontsize=18
)

plt.savefig(
    FIGURES_DIR / "robustness.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 7. GEOGRAPHICAL VISUALIZATION
# ============================================================

print("\n" + "=" * 60)
print("GEOGRAPHICAL VISUALIZATION")
print("=" * 60)

# ------------------------------------------------------------
# Position dictionary
# ------------------------------------------------------------

airport_positions = {}

for node in G_main.nodes():

    if (
        "longitude" in G_main.nodes[node]
        and
        "latitude" in G_main.nodes[node]
    ):

        airport_positions[node] = (

            G_main.nodes[node]["longitude"],

            G_main.nodes[node]["latitude"]
        )

# ------------------------------------------------------------
# Create geographic subgraph
# ------------------------------------------------------------

valid_nodes = [
    node for node in G_main.nodes()
    if node in airport_positions
]

G_geo = G_main.subgraph(valid_nodes)

print("Nodes with coordinates:", G_geo.number_of_nodes())

# ------------------------------------------------------------
# Plot map
# ------------------------------------------------------------

fig = plt.figure(figsize=(20,10))

ax = plt.axes(
    projection=ccrs.PlateCarree()
)

ax.set_global()

# Background
ax.add_feature(
    cfeature.LAND,
    color="lightgray"
)

ax.add_feature(
    cfeature.OCEAN,
    color="aliceblue"
)

ax.add_feature(
    cfeature.COASTLINE,
    linewidth=0.5
)

ax.add_feature(
    cfeature.BORDERS,
    linewidth=0.3
)

# ============================================================
# DRAW ROUTES
# ============================================================

print("Drawing routes...")

for u, v in G_geo.edges():

    if (
        u in airport_positions
        and
        v in airport_positions
    ):

        lon1, lat1 = airport_positions[u]

        lon2, lat2 = airport_positions[v]

        ax.plot(
            [lon1, lon2],
            [lat1, lat2],
            color="steelblue",
            linewidth=0.15,
            alpha=0.08,
            transform=ccrs.Geodetic()
        )

# ============================================================
# DRAW AIRPORTS
# ============================================================

print("Drawing airports...")

node_sizes = []

longitudes = []
latitudes = []

node_colors = []

for node in G_geo.nodes():

    lon, lat = airport_positions[node]

    longitudes.append(lon)

    latitudes.append(lat)

    node_sizes.append(
        degrees[node] * 0.8
    )

    node_colors.append(
        communities[node]
    )

scatter = ax.scatter(
    longitudes,
    latitudes,
    s=node_sizes,
    c=node_colors,
    cmap="tab20",
    alpha=0.8,
    transform=ccrs.PlateCarree()
)

# ============================================================
# TITLE
# ============================================================

plt.title(
    "Global Airport Transportation Network",
    fontsize=22
)

plt.savefig(
    FIGURES_DIR / "geographical_network.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# 8. TOP HUBS VISUALIZATION
# ============================================================

print("\n" + "=" * 60)
print("TOP HUBS")
print("=" * 60)

top_degree = sorted(
    degrees.items(),
    key=lambda x: x[1],
    reverse=True
)[:15]

airports = [x[0] for x in top_degree]

values = [x[1] for x in top_degree]

plt.figure(figsize=(10,6))

plt.barh(
    airports[::-1],
    values[::-1]
)

plt.xlabel("Degree")

plt.title(
    "Top Airport Hubs",
    fontsize=18
)

plt.savefig(
    FIGURES_DIR / "top_hubs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("ALL FIGURES GENERATED")
print("=" * 60)

print("""
Generated figures:

- communities.png
- degree_distribution.png
- loglog_distribution.png
- clustering_vs_degree.png
- spectral_density.png
- robustness.png
- geographical_network.png
- top_hubs.png

All figures saved in:

../figures/
""")

Figures will be saved in:
/home/mzhc13/master/complex-networks-airports/../figures

LOADING NETWORK


FileNotFoundError: [Errno 2] No such file or directory: '/home/mzhc13/master/complex-networks-airports/../data/airport_network.gpickle'